# Raspberry Pi Pico fluorescence spectrometer control

Run these cells from top to bottom in DataSpell after the Pico is connected and `pico/code.py` has been copied to the `CIRCUITPY` drive as `code.py`.

In [ ]:
# Install pyserial only if it is missing from the current DataSpell/Conda environment.

import importlib.util
import subprocess
import sys

if importlib.util.find_spec("serial") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "pyserial"])
else:
    print("pyserial is already installed")

In [ ]:
import csv
import time
from pathlib import Path

import matplotlib.pyplot as plt
import serial
import serial.tools.list_ports

WAVELENGTHS_NM = [415, 445, 480, 515, 555, 590, 630, 680]

In [ ]:
# Find the Pico serial port. Windows ports look like COM5; macOS ports usually look like /dev/cu.usbmodem*.
ports = list(serial.tools.list_ports.comports())

for index, port in enumerate(ports):
    print(f"{index}: {port.device} | {port.description} | {port.hwid}")

pico_candidates = []
for port in ports:
    device = port.device.upper()
    description = port.description.upper()
    hwid = port.hwid.upper()

    if (
        "239A" in hwid
        or "CIRCUITPY" in description
        or "PICO" in description
        or "USB SERIAL DEVICE" in description
        or "USBMODEM" in device
    ):
        pico_candidates.append(port)

# Prefer macOS's callout serial device over /dev/tty.* if both are shown.
pico_candidates.sort(key=lambda port: 0 if port.device.startswith("/dev/cu.") else 1)

if pico_candidates:
    PORT = pico_candidates[0].device
elif ports:
    PORT = ports[0].device
else:
    raise RuntimeError("No serial ports were found. Check that the Pico is connected.")

print("Selected port:", PORT)

In [ ]:
# Open the Pico serial connection.
ser = serial.Serial(PORT, baudrate=115200, timeout=0.2)
ser.dtr = True
ser.rts = True
time.sleep(2)
ser.reset_input_buffer()
print("Connected to", ser.port)

In [ ]:
def _clean_serial_line(raw_line):
    text = raw_line.decode("utf-8", errors="replace").strip()
    return text.lstrip(">").strip()


def read_until_done(timeout=5):
    deadline = time.time() + timeout
    lines = []

    while time.time() < deadline:
        raw_line = ser.readline()
        if not raw_line:
            continue

        line = _clean_serial_line(raw_line)
        if not line:
            continue
        if line == "DONE":
            return lines
        lines.append(line)

    raise TimeoutError("Pico did not reply with DONE before the timeout")


def pico_command(command_text, timeout=5, show=True):
    ser.reset_input_buffer()
    ser.write((command_text.strip() + "\r\n").encode("utf-8"))
    lines = read_until_done(timeout=timeout)

    if show:
        for line in lines:
            print(line)

    errors = [line for line in lines if line.startswith("ERR")]
    if errors:
        raise RuntimeError(errors[0])

    return lines


def _parse_data(lines):
    for line in lines:
        if line.startswith("DATA,"):
            return [int(value) for value in line.split(",")[1:]]
    raise ValueError("No DATA line was returned by the Pico")


def read_spectrum(samples=1):
    lines = pico_command(f"read {samples}", timeout=10, show=False)
    return WAVELENGTHS_NM, _parse_data(lines)


def read_dark(samples=3):
    lines = pico_command(f"dark {samples}", timeout=10, show=False)
    return WAVELENGTHS_NM, _parse_data(lines)


def measure_spectrum(led1=10000, led2=0, samples=3, settle_s=0.1):
    lines = pico_command(
        f"measure {led1} {led2} {samples} {settle_s}",
        timeout=15,
        show=False,
    )
    return WAVELENGTHS_NM, _parse_data(lines)


def plot_spectrum(wavelengths, counts, title="AS7341 spectrum"):
    plt.figure(figsize=(7, 4))
    plt.plot(wavelengths, counts, marker="o")
    plt.xlabel("Wavelength (nm)")
    plt.ylabel("Sensor counts")
    plt.title(title)
    plt.grid(True, alpha=0.3)
    plt.show()

In [ ]:
# Confirm the Pico command interface is responding.
pico_command("status")

In [ ]:
# Set the first-run sensor and LED settings.
pico_command("off")
pico_command("gain 7")
pico_command("integration 100")

In [ ]:
# Take one sensor reading with the LEDs off.
wavelengths, dark_counts = read_dark(samples=3)
print(list(zip(wavelengths, dark_counts)))
plot_spectrum(wavelengths, dark_counts, title="Dark reading")

In [ ]:
# Take a measurement with LED1 on, then automatically switch the LEDs off.
# Start with a modest brightness while the sensor is outside the printed box.
LED1_BRIGHTNESS = 10000
LED2_BRIGHTNESS = 0
SAMPLES = 3

wavelengths, light_counts = measure_spectrum(
    led1=LED1_BRIGHTNESS,
    led2=LED2_BRIGHTNESS,
    samples=SAMPLES,
    settle_s=0.1,
)

corrected_counts = [light - dark for light, dark in zip(light_counts, dark_counts)]

print("Raw:", list(zip(wavelengths, light_counts)))
print("Dark-corrected:", list(zip(wavelengths, corrected_counts)))
plot_spectrum(wavelengths, corrected_counts, title="Dark-corrected fluorescence spectrum")

In [ ]:
# Save the latest measurement as a CSV file next to this notebook.
output_path = Path("data") / "spectrometer_reading.csv"
output_path.parent.mkdir(exist_ok=True)

with output_path.open("w", newline="") as csv_file:
    writer = csv.writer(csv_file)
    writer.writerow(["wavelength_nm", "dark_counts", "light_counts", "corrected_counts"])
    writer.writerows(zip(wavelengths, dark_counts, light_counts, corrected_counts))

print("Saved", output_path.resolve())

In [ ]:
# Manual commands for quick testing.
# pico_command("led1 5000")
# pico_command("read 1")
# pico_command("off")
# pico_command("help")

In [ ]:
# Close the serial connection when you are finished.
pico_command("off")
ser.close()